<a href="https://colab.research.google.com/github/AaronYounger/Quantitative-Finance/blob/main/Simulating_a_Stocks_Price_with_Monte_Carlo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Predicting Stock Prices with Monte Carlo**

In [8]:
# Install Packages
!pip install openbb

INFO: pip is looking at multiple versions of requests-cache to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.8/9.8 MB 57.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 379.4/379.4 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.8/99.8 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.7/97.7 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.5/125.5 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.8/107.8 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.4/43.4 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 73.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 190.3/190.3 kB 9.8 M

In [10]:
# Import Appropriate Packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yfinance as yf
from openbb import obb
obb.user.preferences.output_type = "dataframe"
from scipy.stats import norm

In [15]:
# Download Data
symbol = ['AAPL']
data = obb.equity.price.historical(
    symbol,
    start_date='2021-01-01',
    provider='yfinance'
)
print(data) # Data has correct columns and have correct start and end dates.

                  open        high         low       close     volume  \
date                                                                    
2021-01-04  133.520004  133.610001  126.760002  129.410004  143301900   
2021-01-05  128.889999  131.740005  128.429993  131.009995   97664900   
2021-01-06  127.720001  131.050003  126.379997  126.599998  155088000   
2021-01-07  128.360001  131.630005  127.860001  130.919998  109578200   
2021-01-08  132.429993  132.630005  130.229996  132.050003  105158200   
...                ...         ...         ...         ...        ...   
2026-03-24  250.350006  254.830002  249.550003  251.639999   45152300   
2026-03-25  254.100006  255.000000  251.600006  252.619995   28476700   
2026-03-26  252.119995  257.000000  250.770004  252.889999   41796700   
2026-03-27  253.899994  255.490005  248.070007  248.800003   47842500   
2026-03-30  249.994995  250.839996  245.509995  246.630005   31087414   

            dividend  
date                  
2021

In [17]:
# Calculate Returns
data['Returns'] = data['close'].pct_change()
data.head() # Returns columns is there

,open,high,low,close,volume,dividend,Returns
date,,,,,,,
2021-01-04,133.520004,133.610001,126.760002,129.410004,143301900,0.0,NaN
2021-01-05,128.889999,131.740005,128.429993,131.009995,97664900,0.0,0.012364
2021-01-06,127.720001,131.050003,126.379997,126.599998,155088000,0.0,-0.033662
2021-01-07,128.360001,131.630005,127.860001,130.919998,109578200,0.0,0.034123
2021-01-08,132.429993,132.630005,130.229996,132.050003,105158200,0.0,0.008631


Monte Carlo runs off the principles of Geommetric Brownian Motion.

Formula + Components of GBM:
St = St-1 * exp((u - 0.5o^2) + 0et)

- St = Asset price at time (t)
- St-1 = Asset price at the previous step
- exp = Euler's Value (2.71828)
- u = drift term, representing expected return
- o = volatility of an asset
- et = a random variable drawn from a standard normal distribution

In [18]:
# Create a function for Monte Carlo
def Monte_Carlo(data, num_simulations = 1000, num_days = 22):
  # Inputs
  last_price = data['close'].iloc[-1]
  mu = data['Returns'].mean()
  sigma = data['Returns'].std()

  # Initialize simulation matrix
  simulation_df = np.zeros((num_days, num_simulations))

  # Run simulation
  for simulation in range(num_simulations):
    price_list = [last_price]

    for d in range(num_days):
      price = price_list[-1] * np.exp((mu - 0.5 * sigma**2) + sigma * np.random.normal())
      price_list.append(price)

    simulation_df[:, simulation] = price_list[1:]
  return simulation_df


In [19]:
import ipywidgets as widgets

In [44]:
def interactive_MC(data, num_days=22, num_simulations=1000):
    prices = Monte_Carlo(
        data=data,
        num_days=num_days,
        num_simulations=num_simulations
    )
    terminal_wealth = prices[-1]
    mean_terminal = np.mean(terminal_wealth)
    median_terminal = np.median(terminal_wealth)
    p5 = np.percentile(terminal_wealth, 5)
    p95 = np.percentile(terminal_wealth, 95)
    start_price = data['close'].iloc[-1]
    prob_gain = np.mean(terminal_wealth > start_price)

    plt.figure(figsize=(10, 6))
    plt.title("Monte Carlo Simulation-Nividia Price")

    for i in range(prices.shape[1]):
      color = "green" if terminal_wealth[i] > mean_terminal else "red"
      plt.plot(prices[:, i], color=color, alpha=0.3)

    plt.axhline(y=terminal_wealth.mean(), ls='--', color = "black", label="Mean Final Price")
    plt.axhline(y=start_price, ls='--', color = 'blue', label = 'Starting Price')
    plt.xlabel("Day")
    plt.ylabel("Price")
    plt.show()
    print(f"Mean Final Price: {mean_terminal}")
    print(f"Median Final Price: {median_terminal}")
    print(f"5th Percentile: {p5}")
    print(f"95th Percentile: {p95}")
    print(f"Probability of Gain: {prob_gain}")

In [45]:
widgets.interactive(interactive_MC, data=widgets.fixed(data), num_days=(1,22), num_simulations=(10, 1000,10))

interactive(children=(IntSlider(value=22, description='num_days', max=22, min=1), IntSlider(value=1000, descri…

Why is a monte carlo simulation of stock prices valuable?
1.   Monte Carlo presents a distribution not a prediction.
2.   Answers the questions of: What's the chance I lose money? How bad could it get? What's my worst-case scenario?

Some other practical applications of Monte Carlo is portfolio management and Strategy testing.

Overall, Monte Carlo turns uncertainty into measurable outcomes, quantifies risk and reward, replaces single predictions with probability distributions. Its not about predicting but understasnding the range of possibilities.